# Evaluate pretrained E5-base on Colab

Notebook nay clone repo, patch preset E5, upload `test_cleaned.jsonl`, sau do chay metrics va luu JSON vao Google Drive.

In [ ]:
!nvidia-smi || true
!python -V

In [ ]:
# 1) Clone repo
import os
from pathlib import Path

REPO_URL = "https://github.com/PhamMinhDan/llm_provider_benchmarking_ver2.git"
REPO_DIR = Path('/content/llm_provider_benchmarking')

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists:', REPO_DIR)

%cd /content/llm_provider_benchmarking

In [ ]:
# 2) Install dependencies
!pip install -q -r embedding_project/requirements.txt

In [ ]:
# 3) Mount Google Drive and set output path
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUT_DIR = Path('/content/drive/MyDrive/embedding_project_outputs')
DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Drive output dir:', DRIVE_OUT_DIR)

In [ ]:
# 4) Upload required test file (if missing)
from google.colab import files

DATA_DIR = REPO_DIR / 'embedding_project' / 'data'
TEST_PATH = DATA_DIR / 'test_cleaned.jsonl'

if not TEST_PATH.exists():
    print('Missing test_cleaned.jsonl. Please upload now...')
    uploaded = files.upload()
    if 'test_cleaned.jsonl' not in uploaded:
        raise FileNotFoundError('Ban can upload file test_cleaned.jsonl')
    TEST_PATH.write_bytes(uploaded['test_cleaned.jsonl'])
    print('Uploaded:', TEST_PATH)
else:
    print('Found:', TEST_PATH)

In [ ]:
# 5) Data prechecks
LABELS_PATH = DATA_DIR / 'query_product_labels_cleaned.json'
CSV_PATH = DATA_DIR / 'merged_products_vi_cleaned.csv'

missing = [str(p) for p in [TEST_PATH, LABELS_PATH, CSV_PATH] if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Thieu file dau vao:\n' + '\n'.join(missing) + '\n\n'
        'Hay upload/copy cac file con thieu vao embedding_project/data/ truoc khi chay eval.'
    )

print('All required files exist.')
print(' -', TEST_PATH)
print(' -', LABELS_PATH)
print(' -', CSV_PATH)

In [ ]:
# 6) Patch model_presets.py to add e5-base preset (idempotent)
from pathlib import Path

preset_path = REPO_DIR / 'embedding_project' / 'scripts' / 'model_presets.py'
text = preset_path.read_text(encoding='utf-8')

if '"e5-base"' not in text:
    insert_after = '    "bge-m3": EmbeddingModelPreset(\n'
    idx = text.find(insert_after)
    if idx == -1:
        raise RuntimeError('Cannot locate bge-m3 preset block in model_presets.py')

    bge_end = text.find('    ),\n}', idx)
    if bge_end == -1:
        raise RuntimeError('Cannot locate end of PRESETS block')

    e5_block = '''    "e5-base": EmbeddingModelPreset(
        name="e5-base",
        base_model="intfloat/multilingual-e5-base",
        final_subdir="e5_base_finetuned_final",
        run_name="e5-base-vi-semantic-search",
        max_seq_length=512,
        epochs=1,
        batch_size=8,
        learning_rate=1e-5,
        warmup_ratio=0.1,
        fp16_default=True,
        trust_remote_code=False,
        metrics_filename="metrics_e5_base.json",
    ),
'''
    text = text[:bge_end + len('    ),\n')] + e5_block + text[bge_end + len('    ),\n'):]
    preset_path.write_text(text, encoding='utf-8')
    print('Added e5-base preset.')
else:
    print('e5-base preset already exists.')

In [ ]:
# 7) Patch evaluate_embedding_model.py for e5 query/passage prefixes (idempotent)
eval_path = REPO_DIR / 'embedding_project' / 'scripts' / 'evaluate_embedding_model.py'
text = eval_path.read_text(encoding='utf-8')

text = text.replace(
    'parser.add_argument("--preset", choices=["minilm", "bge-m3"], default="minilm")',
    'parser.add_argument("--preset", choices=["minilm", "bge-m3", "e5-base"], default="minilm")'
)

marker = 'result["k"] = args.k\n'
e5_logic = (
    '    if args.preset == "e5-base":\n'
    '        # E5 retrieval convention: query/passage prefixes improve quality.\n'
    '        query_texts = [f"query: {q}" for q in query_texts]\n'
    '        corpus_texts = [f"passage: {t}" for t in corpus_texts]\n'
)
if e5_logic not in text:
    text = text.replace(marker, marker + '\n' + e5_logic)

eval_path.write_text(text, encoding='utf-8')
print('Patched evaluate_embedding_model.py')

In [ ]:
# 8) Run pretrained E5 evaluation (k=10)
OUT_JSON = DRIVE_OUT_DIR / 'metrics_e5_base.json'

!python embedding_project/scripts/evaluate_embedding_model.py \
    --preset e5-base \
    --only-pretrained \
    --k 10 \
    --test-jsonl embedding_project/data/test_cleaned.jsonl \
    --labels-json embedding_project/data/query_product_labels_cleaned.json \
    --products-csv embedding_project/data/merged_products_vi_cleaned.csv \
    --output {OUT_JSON}

In [ ]:
# 9) Print saved metrics JSON
import json

metrics = json.loads(Path(OUT_JSON).read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('\nSaved to:', OUT_JSON)

In [ ]:
# Optional smoke check
from sentence_transformers import SentenceTransformer
m = SentenceTransformer('intfloat/multilingual-e5-base')
vec = m.encode(['query: giay chay bo nam nhe', 'query: vay nu du tiec'])
print('Emb shape:', vec.shape)